In [ ]:
from xinghe.spark import new_spark_session, read_any_path


config = {
    "spark_conf_name": "spark_8",
    "skip_success_check": True,
    "spark.yarn.queue": "root.warehouse",
}

spark = new_spark_session("LINFENG_OPENSEARCH_CHUNK_IMPORT", config)
spark.sparkContext.setLogLevel("ERROR")

spark

In [ ]:
INPUT_PATH = "s3://web-parse-hw60p/linfeng/agentic_search/oa_chunk_data/v3/data/"
raw_df = read_any_path(spark, INPUT_PATH, config)

raw_df.printSchema()
raw_df.select("value").show(3, truncate=False)

In [ ]:
from datetime import datetime, timezone
import json
import math
import socket

from opensearchpy import OpenSearch
from pyspark.sql.functions import col, from_json, lower, transform, udf
from pyspark.sql.types import LongType, MapType, StringType, StructField, StructType

# --- OpenSearch 连接配置（按实际环境修改）---
OS_HOSTS = [{"host": "pjlab-dataproducer-opensearch-api.dataops-cloud.d.pjlab.org.cn", "port": 80}]
OS_AUTH = ("", "")
OS_INDEX = "agentic_search_chunk_v2"

# --- 极限写入参数 ---
TOTAL_DATA_GB = 10080
SHARD_SIZE_GB = 40
NUM_SHARDS = math.ceil(TOTAL_DATA_GB / SHARD_SIZE_GB)

BULK_SIZE = 10_000
PARTITIONS = 100
# 建索引策略：测试期默认删掉同名索引后重建；如果想复用已有索引，改成 False。
RECREATE_INDEX = True
INDEX_DELETE_WAIT_SECONDS = 30
# 先抽样验证：默认只写 10 条。全量导入时改成 None 或 0。
TEST_SAMPLE_ROWS = None
WRITE_PARTITIONS = max(1, min(PARTITIONS, TEST_SAMPLE_ROWS or PARTITIONS))
OS_CONN = dict(hosts=OS_HOSTS, http_auth=OS_AUTH, use_ssl=False, verify_certs=False)

client = OpenSearch(**OS_CONN, timeout=60)


def validate_os_hosts(hosts):
    if not hosts:
        raise ValueError("OS_HOSTS 不能为空")
    for item in hosts:
        host = str((item or {}).get("host") or "").strip()
        port = int((item or {}).get("port") or 0)
        if not host:
            raise ValueError(f"OS_HOSTS 配置缺少 host: {item!r}")
        if host == "your-opensearch-host":
            raise ValueError("OS_HOSTS 仍是占位符 your-opensearch-host，请改成真实 OpenSearch 域名或 IP")
        if port <= 0:
            raise ValueError(f"OS_HOSTS 配置缺少有效 port: {item!r}")
        try:
            socket.getaddrinfo(host, port, socket.AF_UNSPEC, socket.SOCK_STREAM)
        except socket.gaierror as exc:
            raise ValueError(
                f"无法解析 OpenSearch 主机名 {host!r}:{port}，请检查 OS_HOSTS、DNS、VPN 或内网连通性"
            ) from exc


def probe_opensearch(client):
    try:
        info = client.info()
    except Exception as exc:
        raise RuntimeError(f"OpenSearch 连通性探测失败: {exc}") from exc
    cluster_name = info.get("cluster_name") or ""
    version = ((info.get("version") or {}).get("number")) or ""
    print(f"OpenSearch reachable: cluster={cluster_name} version={version}")
    return info


def wait_until_index_absent(client, index, timeout_seconds=30):
    import time

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if not client.indices.exists(index=index):
            return True
        time.sleep(1)
    return not client.indices.exists(index=index)


def utc_millis_to_iso_z(ms):
    if ms is None:
        return None
    ms = int(ms)
    if ms <= 0:
        return None
    dt = datetime.fromtimestamp(ms / 1000.0, tz=timezone.utc)
    return dt.strftime("%Y-%m-%dT%H:%M:%S.") + f"{dt.microsecond // 1000:03d}Z"


def parse_extra_info(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return {str(k): None if v is None else str(v) for k, v in value.items()}
    s = str(value).strip()
    if not s:
        return None
    try:
        obj = json.loads(s)
    except Exception:
        return None
    if not isinstance(obj, dict):
        return None
    out = {}
    for k, v in obj.items():
        if v is None:
            out[str(k)] = None
        elif isinstance(v, (dict, list)):
            out[str(k)] = json.dumps(v, ensure_ascii=False)
        else:
            out[str(k)] = str(v)
    return out


utc_millis_to_iso_z_udf = udf(utc_millis_to_iso_z, StringType())
parse_extra_info_udf = udf(parse_extra_info, MapType(StringType(), StringType(), True))


In [ ]:
# --- 1. 创建索引（按当前样例字段）---
validate_os_hosts(OS_HOSTS)
probe_opensearch(client)

index_exists = client.indices.exists(index=OS_INDEX)
print(f"index exists before create: {index_exists}")
if index_exists and RECREATE_INDEX:
    client.indices.delete(index=OS_INDEX, params={"ignore_unavailable": "true"})
    deleted = wait_until_index_absent(client, OS_INDEX, timeout_seconds=INDEX_DELETE_WAIT_SECONDS)
    if not deleted:
        raise RuntimeError(f"删除索引后仍检测到存在: {OS_INDEX}")
    print(f"index deleted: {OS_INDEX}")

BULK_LOAD_SETTINGS = {
    "number_of_shards": NUM_SHARDS,
    "number_of_replicas": 0,
    "refresh_interval": "-1",
    "translog.durability": "async",
    "translog.sync_interval": "120s",
    "translog.flush_threshold_size": "2048mb",
    "index.merge.scheduler.max_thread_count": 4,
    "index.merge.scheduler.max_merge_count": 16,
    "index.codec": "best_compression",
}

if client.indices.exists(index=OS_INDEX):
    print(f"index already exists, reuse existing index: {OS_INDEX}")
else:
    client.indices.create(
        index=OS_INDEX,
        body={
            "settings": BULK_LOAD_SETTINGS,
            "mappings": {
                "dynamic": "strict",
                "properties": {
                    "record_type": {"type": "keyword"},
                    "chunk_id": {"type": "keyword"},
                    "doc_id": {"type": "keyword"},
                    "title": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "content": {"type": "text"},
                    "abstract": {"type": "text"},
                    "url": {
                        "type": "keyword",
                        "doc_values": False,
                        "ignore_above": 4096,
                    },
                    "category": {"type": "keyword"},
                    "source_type": {"type": "keyword"},
                    "lang": {"type": "keyword"},
                    "job_id": {"type": "keyword"},
                    "task_id": {"type": "keyword"},
                    "extra_info": {"type": "object", "dynamic": True},
                    "created_time": {
                        "type": "date",
                        "format": "strict_date_optional_time||epoch_millis",
                    },
                    "updated_time": {
                        "type": "date",
                        "format": "strict_date_optional_time||epoch_millis",
                    },
                    "offset": {"type": "long"},
                    "page_no": {"type": "integer"},
                    "chunk_no": {"type": "integer"},
                    "publication_published_date": {
                        "type": "date",
                        "format": "strict_date_optional_time||yyyy-MM-dd||yyyy-MM||yyyy",
                    },
                    "publication_published_year": {"type": "integer"},
                    "metadata_type": {"type": "keyword"},
                    "publication_venue_type": {"type": "keyword"},
                    "primary_topic": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "primary_topic_subfield": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "primary_topic_field": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "primary_topic_domain": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "author": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "publication_venue_name_unified": {
                        "type": "text",
                        "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                    },
                    "access_is_oa": {"type": "keyword"},
                    "citation_count": {"type": "integer"},
                    "influential_citation_count": {"type": "integer"},
                },
            },
        },
    )
    print(f"index created: {OS_INDEX}, shards={NUM_SHARDS}")


In [ ]:
# --- 2. 准备数据 ---
# 这里直接使用当前输入的目标字段名。
raw_schema = """
record_type STRING,
chunk_id STRING,
doc_id STRING,
title STRING,
content STRING,
abstract STRING,
url STRING,
category STRING,
source_type STRING,
lang STRING,
job_id STRING,
task_id STRING,
extra_info MAP<STRING,STRING>,
created_time STRING,
updated_time STRING,
offset BIGINT,
page_no BIGINT,
chunk_no BIGINT,
publication_published_date STRING,
publication_published_year BIGINT,
metadata_type STRING,
publication_venue_type STRING,
primary_topic STRING,
primary_topic_subfield STRING,
primary_topic_field STRING,
primary_topic_domain STRING,
author ARRAY<STRING>,
publication_venue_name_unified STRING,
access_is_oa STRING,
citation_count BIGINT,
influential_citation_count BIGINT
"""

parsed_df = raw_df.select(from_json(col("value"), raw_schema).alias("d")).select("d.*")
source_df = parsed_df.limit(TEST_SAMPLE_ROWS) if TEST_SAMPLE_ROWS else parsed_df

df = (
    source_df.select(
        col("record_type"),
        col("chunk_id"),
        col("doc_id"),
        lower(col("title")).alias("title"),
        col("content"),
        col("abstract"),
        col("url"),
        col("category"),
        col("source_type"),
        col("lang"),
        col("job_id"),
        col("task_id"),
        parse_extra_info_udf(col("extra_info")).alias("extra_info"),
        col("created_time"),
        col("updated_time"),
        col("offset"),
        col("page_no"),
        col("chunk_no"),
        col("publication_published_date"),
        col("publication_published_year"),
        col("metadata_type"),
        lower(col("publication_venue_type")).alias("publication_venue_type"),
        lower(col("primary_topic")).alias("primary_topic"),
        lower(col("primary_topic_subfield")).alias("primary_topic_subfield"),
        lower(col("primary_topic_field")).alias("primary_topic_field"),
        lower(col("primary_topic_domain")).alias("primary_topic_domain"),
        transform(col("author"), lambda x: lower(x)).alias("author"),
        lower(col("publication_venue_name_unified")).alias("publication_venue_name_unified"),
        col("access_is_oa"),
        col("citation_count"),
        col("influential_citation_count"),
    )
    .repartition(WRITE_PARTITIONS)
)

prepared_rows = df.count()
print(f"rows prepared for write: {prepared_rows}")
preview_docs = [row.asDict(recursive=True) for row in df.limit(TEST_SAMPLE_ROWS or 10).collect()]
print(json.dumps(preview_docs, ensure_ascii=False, indent=2))

df.printSchema()
df.show(3, truncate=False)


In [ ]:
# --- 3. 写入 ---
def _chunked(iterable, size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch


def write_partition(partition):
    import json
    import logging
    import time

    log = logging.getLogger("os_writer")
    c = OpenSearch(
        **OS_CONN,
        timeout=180,
        max_retries=2,
        retry_on_timeout=True,
        http_compress=True,
    )

    action_line = json.dumps({"index": {"_index": OS_INDEX}}, ensure_ascii=False)

    def _ndjson(docs):
        for d in docs:
            yield action_line
            yield json.dumps(d, ensure_ascii=False)

    def send(docs, retries=3):
        body = "\n".join(_ndjson(docs)) + "\n"
        resp = None
        for attempt in range(retries + 1):
            try:
                resp = c.bulk(
                    body=body,
                    filter_path="errors,items.index.error.type",
                    request_timeout=180,
                )
                if not resp.get("errors"):
                    return len(docs), 0
            except Exception:
                resp = None
            if attempt < retries:
                time.sleep(min(2 ** attempt * 5, 30))
        fails = sum(
            1 for it in (resp or {}).get("items", [])
            if (it.get("index") or {}).get("error")
        )
        return len(docs) - fails, fails

    rows = (
        {k: v for k, v in row.asDict(recursive=True).items() if v is not None}
        for row in partition
    )

    written = failed = 0
    t0 = time.time()
    for batch in _chunked(rows, BULK_SIZE):
        ok, fail = send(batch)
        written += ok
        failed += fail

    elapsed = max(time.time() - t0, 1)
    log.warning(
        "done: %d written, %d failed, %.0f docs/s, %.1f s",
        written,
        failed,
        written / elapsed,
        elapsed,
    )


In [ ]:
df.foreachPartition(write_partition)
print(f"bulk import finished, rows attempted: {prepared_rows}")


In [ ]:
# --- 4. 回查验证 ---
client.indices.refresh(index=OS_INDEX)
count_resp = client.count(index=OS_INDEX)
print(f"indexed docs after refresh: {count_resp['count']}")

verify_resp = client.search(
    index=OS_INDEX,
    body={
        "size": 10,
        "sort": ["_doc"],
        "query": {"match_all": {}},
    },
)
verify_docs = [hit.get("_source", {}) for hit in verify_resp.get("hits", {}).get("hits", [])]
print(json.dumps(verify_docs, ensure_ascii=False, indent=2))


In [ ]:
# --- 5. 切换为只读查询优化设置 ---
if TEST_SAMPLE_ROWS:
    print("测试模式：已跳过 force merge 和只读优化设置。确认样本无误后，将 TEST_SAMPLE_ROWS 改成 None 或 0 再全量导入。")
else:
    client.indices.refresh(index=OS_INDEX, params={"wait_for_completion": "false"})
    print("refresh 已在后台启动，通过 GET /_tasks?actions=*refresh&detailed 观察进度")

    client.indices.forcemerge(
        index=OS_INDEX,
        max_num_segments=1,
        flush=True,
        params={"wait_for_completion": "false"},
    )
    print("force merge 已在后台启动，通过 GET /_cat/segments/<index>?v 观察 segment 数量收敛")

    READONLY_SETTINGS = {
        "index": {
            "number_of_replicas": 1,
            "refresh_interval": "30s",
            "translog.durability": "request",
            "merge.scheduler.max_thread_count": 1,
            "merge.scheduler.max_merge_count": 1,
        }
    }

    client.indices.put_settings(index=OS_INDEX, body=READONLY_SETTINGS)
    print("readonly settings applied")


In [ ]:
# 大索引 force merge 可能超过网关超时，默认改为异步启动。
FORCE_MERGE_ASYNC = True
FORCE_MERGE_TASK_ID = None
# --- 5A. 启动 force merge ---
if TEST_SAMPLE_ROWS:
    print("测试模式：已跳过 force merge 和只读优化设置。确认样本无误后，将 TEST_SAMPLE_ROWS 改成 None 或 0 再全量导入。")
else:
    client.indices.refresh(index=OS_INDEX)
    print("refresh 已完成")

    if FORCE_MERGE_ASYNC:
        force_merge_resp = client.indices.forcemerge(
            index=OS_INDEX,
            max_num_segments=1,
            flush=True,
            params={"wait_for_completion": "false"},
        )
        FORCE_MERGE_TASK_ID = force_merge_resp.get("task")
        print(f"force merge 已异步启动，task_id={FORCE_MERGE_TASK_ID}")
        print("merge 完成后再执行下一个单元应用只读优化设置")
    else:
        client.indices.forcemerge(
            index=OS_INDEX,
            max_num_segments=1,
            flush=True,
        )
        FORCE_MERGE_TASK_ID = None
        print("force merge 已同步完成，可直接执行下一个单元")

In [ ]:
# --- 5B. merge 完成后执行只读优化设置 ---
READONLY_SETTINGS = {
    "index": {
        "number_of_replicas": 1,
        "refresh_interval": "30s",
        "translog.durability": "request",
        "merge.scheduler.max_thread_count": 1,
        "merge.scheduler.max_merge_count": 1,
    }
}

if TEST_SAMPLE_ROWS:
    print("测试模式：跳过只读优化设置")
elif FORCE_MERGE_ASYNC and FORCE_MERGE_TASK_ID:
    try:
        task_resp = client.tasks.get(task_id=FORCE_MERGE_TASK_ID)
        completed = bool(task_resp.get("completed"))
        print(json.dumps(task_resp, ensure_ascii=False, indent=2))
        if not completed:
            raise RuntimeError(f"force merge 尚未完成，请稍后重试本单元。task_id={FORCE_MERGE_TASK_ID}")
    except Exception as exc:
        msg = str(exc)
        if "404" in msg or "resource_not_found_exception" in msg:
            print(f"未在 Tasks API 中找到 task_id={FORCE_MERGE_TASK_ID}，按已完成处理")
        else:
            raise
    client.indices.put_settings(index=OS_INDEX, body=READONLY_SETTINGS)
    print("readonly settings applied")
else:
    client.indices.put_settings(index=OS_INDEX, body=READONLY_SETTINGS)
    print("readonly settings applied")
